# LinkedIn Skills

In this chapter, **skills** are standardised LinkedIn skill labels drawn from members' profiles. LinkedIn groups its skills into broader categories and uses them to describe the capabilities associated with a country or industry. These measures describe the skills represented on LinkedIn; they are not counts of workers, vacancies, or training completions.

The **Skills Genome** is an ordered list of the skills that are most characteristic of an entity, such as a country-industry pair. LinkedIn calculates this list with term frequency-inverse document frequency (TF-IDF), which gives more weight to distinctive skills and down-ranks skills that are common across many entities. **Pooled skills** use skills added across all years in the reporting period, while **flow skills** use skills added during an individual year.

In [1]:
from pathlib import Path

import altair as alt
import attaviz
import pandas as pd

attaviz.enable()
alt.data_transformers.enable("vegafusion")


def find_project_root(marker="pyproject.toml"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "LinkedIn"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(exist_ok=True)
SKILLS_FILE = (
    DATA_PATH
    / "LinkedIn Skills Genome and Penetration"
    / "Skills Genome and Skills Pen 2026.xlsx"
)

WEST_AFRICA = ["Ghana", "Nigeria"]
COMPARATORS = ["India", "Kenya", "South Africa"]
COUNTRIES = WEST_AFRICA + COMPARATORS
COUNTRY_COLOURS = dict(zip(COUNTRIES, attaviz.CATEGORICAL))

SKILL_ORDER = [
    "Soft Skills",
    "Tech Skills",
    "Business Skills",
    "Disruptive Tech Skills",
    "Green Skills",
]
TECH_MEDIA_INDUSTRY = "Technology, Information and Media"
TOP_SKILLS = 10
BENCHMARK = 1.0
SKILLS_NOTE = (
    "Source: LinkedIn Economic Graph, Skills Genome and Skills Penetration 2026."
)

In [2]:
def read_skills(sheet_name, header, names=None, countries=None):
    """Read one Skills Genome sheet and return a tidy frame."""
    data = pd.read_excel(SKILLS_FILE, sheet_name=sheet_name, header=header).dropna(
        axis="columns", how="all"
    )
    if names is not None:
        data = data.set_axis(names, axis="columns")
    else:
        data = data.loc[
            :, [c for c in data.columns if not str(c).startswith("Unnamed")]
        ]
    return (
        data.loc[lambda d: d["Country"].ne("Country")]
        .dropna(subset=data.columns.tolist())
        .assign(
            **{
                c: lambda d, c=c: d[c].astype(str).str.strip()
                for c in data.columns
                if pd.api.types.is_string_dtype(data[c])
            }
        )
        .loc[lambda d: d["Country"].isin(countries) if countries else slice(None)]
        .drop_duplicates()
        .reset_index(drop=True)
    )


def read_comparators():
    """Return the workbook's comparator list, one row per country."""
    return (
        pd.read_excel(SKILLS_FILE, sheet_name="Ref - Country Comparators", header=3)
        .dropna(axis="columns", how="all")
        .dropna(subset=["Country"])
        .set_index("Country")
    )


def skill_scale(domain):
    """Use consistent country colours across all skills charts."""
    domain = list(domain)
    return alt.Scale(domain=domain, range=[COUNTRY_COLOURS[c] for c in domain])

## Skills penetration

**Skills penetration** measures how strongly broad skill groups appear in an entity's characteristic skill profile. At a high level, it is the share of an entity's top 50 skills that belongs to a given skill group. To produce comparable country-industry estimates, LinkedIn identifies representative skills for each industry-occupation pair, assigns them to skill groups, and then aggregates the results.

The charts use **relative skills penetration**: national penetration divided by a comparable global benchmark that accounts for the industry's occupational mix. A value of 1.0 matches the global benchmark, a value above 1.0 indicates higher penetration, and a value below 1.0 indicates lower penetration.

### Pooled by country (2017–2025)

In [3]:
comparators = read_comparators()
skill_penetration = read_skills("3A - SPP Ctry", header=5, countries=COUNTRIES).assign(
    **{c: lambda d, c=c: pd.to_numeric(d[c]) for c in ["Average", "Global", "Relative"]}
)
skill_penetration.to_csv(PROCESSED_PATH / "skill_penetration_country.csv", index=False)


def plot_country_penetration(data, width=420, height=280):
    """Plot relative skill-group penetration for the selected countries."""
    countries = [c for c in COUNTRIES if c in set(data["Country"])]
    highlight = alt.selection_point(fields=["Country"], bind="legend", name="pick")
    top = data["Relative"].max() * 1.08

    bars = (
        alt.Chart(data)
        .mark_bar()
        .encode(
            x=alt.X(
                "Skill:N", title=None, sort=SKILL_ORDER, axis=alt.Axis(labelAngle=-30)
            ),
            xOffset=alt.XOffset("Country:N", sort=countries),
            y=alt.Y(
                "mean(Relative):Q",
                title="Relative penetration",
                scale=alt.Scale(domain=[0, top]),
            ),
            color=alt.Color("Country:N", scale=skill_scale(countries), title=None),
            opacity=alt.when(highlight).then(alt.value(1)).otherwise(alt.value(0.25)),
            tooltip=[
                "Country:N",
                "Skill:N",
                alt.Tooltip("mean(Relative):Q", format=".2f", title="Relative"),
            ],
        )
        .add_params(highlight)
    )
    parity = (
        alt.Chart(data)
        .transform_aggregate(_rows="count()")
        .mark_rule(color=attaviz.REFERENCE, strokeDash=[4, 4])
        .encode(y=alt.datum(BENCHMARK))
    )
    chart = alt.layer(bars, parity).properties(
        width=width,
        height=height,
        title="Relative skill-group penetration against the global benchmark",
    )
    return attaviz.add_caption(
        chart,
        ["Dashed line is parity with the global benchmark (1.0).", SKILLS_NOTE],
        align="left",
    )


plot_country_penetration(skill_penetration)

alt.VConcatChart(...)

### Pooled by country and industry (2017–2025)

This view applies the same relative penetration measure within industries. Select an industry from the menu to compare the selected countries.

In [4]:
industry_penetration = read_skills(
    "3B - SPP Ctry Ind", header=5, countries=COUNTRIES
).assign(
    **{c: lambda d, c=c: pd.to_numeric(d[c]) for c in ["Average", "Global", "Relative"]}
)
industry_penetration.to_csv(
    PROCESSED_PATH / "skill_penetration_industry.csv", index=False
)


def plot_industry_penetration(data, width=420, height=280):
    """Plot relative penetration by skill group and selected industry."""
    countries = [c for c in COUNTRIES if c in set(data["Country"])]
    industries = sorted(data["Industry"].unique())
    pick = alt.selection_point(
        fields=["Industry"],
        bind=alt.binding_select(options=industries, name="Industry  "),
        value=[{"Industry": industries[0]}],
    )
    top = data["Relative"].max() * 1.08
    base = alt.Chart(data).transform_filter(pick)
    bars = base.mark_bar().encode(
        x=alt.X("Skill:N", title=None, sort=SKILL_ORDER, axis=alt.Axis(labelAngle=-30)),
        xOffset=alt.XOffset("Country:N", sort=countries),
        y=alt.Y(
            "Relative:Q", title="Relative penetration", scale=alt.Scale(domain=[0, top])
        ),
        color=alt.Color("Country:N", scale=skill_scale(countries), title=None),
        tooltip=[
            "Country:N",
            "Industry:N",
            "Skill:N",
            alt.Tooltip("Relative:Q", format=".2f"),
            alt.Tooltip("Average:Q", format=".3f", title="National"),
            alt.Tooltip("Global:Q", format=".3f", title="Global"),
        ],
    )
    parity = (
        base.transform_aggregate(_rows="count()")
        .mark_rule(color=attaviz.REFERENCE, strokeDash=[4, 4])
        .encode(y=alt.datum(BENCHMARK))
    )
    chart = (
        alt.layer(bars, parity)
        .add_params(pick)
        .properties(
            width=width,
            height=height,
            title="Relative skill-group penetration by industry",
        )
    )
    return attaviz.add_caption(
        chart,
        ["Dashed line is parity with the global benchmark (1.0).", SKILLS_NOTE],
        align="left",
    )


plot_industry_penetration(industry_penetration)

alt.VConcatChart(...)

### Relative importance in gender skill profiles

**Relative importance** measures how strongly each skill group characterises women's or men's profiles within an industry. It uses TF-IDF scores, so it highlights skills that are distinctive rather than simply common.

For each gender-industry pair, LinkedIn takes the top 30 characteristic skills, weights each skill by its share of the total TF-IDF score, and aggregates them into broader skill groups. A higher value means that the skill group contributes more to what makes that profile distinctive; it does not measure how many people report the skill.

In [5]:
gender_skills = read_skills(
    "3B - SPP Ctry Ind Gen", header=5, countries=COUNTRIES
).assign(
    **{
        c: lambda d, c=c: pd.to_numeric(d[c])
        for c in ["Relative Importance", "Skill Group Penetration"]
    },
    Gender=lambda d: d["Gender"].str.title(),
)
gender_skills.to_csv(PROCESSED_PATH / "gender_skill_profile.csv", index=False)


def plot_gender_profile(data, width=200, height=170):
    """Compare female and male relative importance by skill group."""
    wide = (
        data.pivot_table(
            index=["Country", "Industry", "Skill"],
            columns="Gender",
            values="Relative Importance",
            aggfunc="mean",
        )
        .reset_index()
        .rename_axis(columns=None)
    )
    paired = wide.dropna(subset=["Female", "Male"])
    industries = sorted(wide["Industry"].unique())
    countries = [c for c in COUNTRIES if c in set(wide["Country"])]
    default_industry = wide.groupby("Industry").size().idxmax()
    pick = alt.selection_point(
        fields=["Industry"],
        bind=alt.binding_select(options=industries, name="Industry  "),
        value=[{"Industry": default_industry}],
    )
    top = wide[["Female", "Male"]].max().max() * 1.08
    x_title = "Relative importance (TF-IDF weighted)"
    skills = [s for s in SKILL_ORDER if s in set(wide["Skill"])]

    def panel(country, first):
        y = alt.Y(
            "Skill:N",
            title=None,
            scale=alt.Scale(domain=skills),
            axis=alt.Axis(labels=first),
        )
        x_scale = alt.Scale(domain=[0, top])
        connector = (
            alt.Chart(paired.loc[paired["Country"].eq(country)])
            .transform_filter(pick)
            .mark_rule(color=attaviz.GREY_300, strokeWidth=2)
            .encode(y=y, x=alt.X("Female:Q", title=x_title, scale=x_scale), x2="Male:Q")
        )
        dots = (
            alt.Chart(wide.loc[wide["Country"].eq(country)])
            .transform_filter(pick)
            .transform_fold(["Female", "Male"], as_=["Gender", "Relative Importance"])
            .transform_filter("isValid(datum['Relative Importance'])")
            .mark_point(size=90, filled=True, opacity=1)
            .encode(
                y=y,
                x=alt.X("Relative Importance:Q", title=x_title, scale=x_scale),
                color=alt.Color(
                    "Gender:N",
                    scale=alt.Scale(
                        domain=["Female", "Male"],
                        range=[attaviz.GENDER["female"], attaviz.GENDER["male"]],
                    ),
                    title=None,
                ),
                tooltip=[
                    "Country:N",
                    "Industry:N",
                    "Skill:N",
                    "Gender:N",
                    alt.Tooltip("Relative Importance:Q", format=".3f"),
                ],
            )
        )
        return (
            alt.layer(connector, dots)
            .add_params(pick)
            .properties(width=width, height=height, title=country)
        )

    chart = alt.hconcat(
        *(panel(c, i == 0) for i, c in enumerate(countries)), spacing=14
    ).properties(title="Relative importance of skill groups by gender")
    return attaviz.add_caption(
        chart,
        [
            "A skill group with one dot is reported for that gender only; a missing row is not reported. Nigeria is absent from this sheet.",
            SKILLS_NOTE,
        ],
        align="left",
    )


plot_gender_profile(gender_skills)

/var/folders/q1/wt8mfyzs73l2r5977rk_mkxm0000gn/T/ipykernel_49463/3473648917.py:97: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_gender_profile(gender_skills)


alt.VConcatChart(...)

## Top skills in Technology, Information and Media

LinkedIn's latest release expands each Skills Genome list from 10 to 30 skills. To keep the tables readable and comparable with earlier releases, the following views show only ranks 1–10 for the **Technology, Information and Media** sector. Shading identifies skills that recur across countries or years within each table.

In [6]:
VIRIDIS = [
    "#440154",
    "#482878",
    "#3e4989",
    "#31688e",
    "#26828e",
    "#1f9e89",
    "#35b779",
    "#6ece58",
    "#b5de2b",
    "#fde725",
]
UNIQUE_FILL = "#eaf2f8"


def ranked_skill_table(data, columns, caption):
    """Format a rank-by-country or rank-by-year skills table."""
    grid = (
        data.sort_values([columns, "Skill Rank"])
        .pivot(index="Skill Rank", columns=columns, values="Skill")
        .rename_axis(index="Rank", columns=None)
    )
    counts = data.groupby("Skill")[columns].nunique()
    recurring = list(counts[counts > 1].index)
    colours = {skill: VIRIDIS[i % len(VIRIDIS)] for i, skill in enumerate(recurring)}

    def shade(skill):
        if pd.isna(skill):
            return ""
        if skill in colours:
            return f"background-color: {colours[skill]}; color: white"
        return f"background-color: {UNIQUE_FILL}"

    return (
        grid.style.map(shade)
        .set_caption(caption)
        .set_properties(
            **{"font-size": "11px", "padding": "4px 6px", "border": "1px solid white"}
        )
        .set_table_styles([{"selector": "th", "props": [("font-size", "11px")]}])
    )

### Pooled skills (2017–2025)

Pooled skills use skills added throughout the full reporting period. The table includes every selected country for which LinkedIn reports this sector.

In [7]:
pooled_skills = (
    read_skills(
        "2A - SGP Ctry Ind",
        header=3,
        names=["Country", "Industry", "Skill", "Skill Rank"],
        countries=COUNTRIES,
    )
    .assign(**{"Skill Rank": lambda d: pd.to_numeric(d["Skill Rank"])})
    .loc[
        lambda d: d["Industry"].eq(TECH_MEDIA_INDUSTRY) & d["Skill Rank"].le(TOP_SKILLS)
    ]
)
pooled_skills.to_csv(
    PROCESSED_PATH / "pooled_skills_technology_information_media_top10.csv",
    index=False,
)
available_countries = [c for c in COUNTRIES if c in set(pooled_skills["Country"])]
display(
    ranked_skill_table(
        pooled_skills.assign(
            Country=lambda d: pd.Categorical(
                d["Country"], categories=available_countries, ordered=True
            )
        ),
        columns="Country",
        caption="Top 10 pooled skills in Technology, Information and Media",
    )
)

,Ghana,Nigeria,India,Kenya,South Africa
Rank,,,,,
1,React.js,Virtual Assistance,Core Java,Swahili,Afrikaans
2,Front-End Development,React.js,Amazon Web Services (AWS),Virtual Assistance,Telecommunications
3,Node.js,Virtual Administrative Support,Spring Boot,React.js,Broadcasting
4,Web Development,Search Engine Optimization (SEO),Java,Search Engine Optimization (SEO),Wireless Technologies
5,Broadcasting,Front-End Development,SQL,Python (Programming Language),Voice over IP (VoIP)
6,JavaScript,Web Content Writing,Jenkins,Agile Application Development,Software Development Life Cycle (SDLC)
7,Telecommunications,Responsive Web Design,REST APIs,Editing,IT Integration
8,Python (Programming Language),Creative Writing,C (Programming Language),Virtual Administrative Support,Television
9,Cascading Style Sheets (CSS),User Interface Design,Data Structures,Exceeding Customer Expectations,Managed Services


### Flow skills by year

Flow skills use only the skills added in each year, revealing how the sector's most characteristic new skills change over time. Each country table shows the years available in the workbook and is limited to the top 10 ranks.

In [8]:
flow_skills = (
    read_skills(
        "2B - SGF Ctry Ind Yr",
        header=3,
        names=["Country", "Year", "Industry", "Skill", "Skill Rank"],
        countries=COUNTRIES,
    )
    .assign(
        Year=lambda d: pd.to_numeric(d["Year"]).astype(int),
        **{"Skill Rank": lambda d: pd.to_numeric(d["Skill Rank"])},
    )
    .loc[
        lambda d: d["Industry"].eq(TECH_MEDIA_INDUSTRY) & d["Skill Rank"].le(TOP_SKILLS)
    ]
)
flow_skills.to_csv(
    PROCESSED_PATH / "flow_skills_technology_information_media_top10.csv",
    index=False,
)

for country in [c for c in COUNTRIES if c in set(flow_skills["Country"])]:
    display(
        ranked_skill_table(
            flow_skills.loc[flow_skills["Country"].eq(country)],
            columns="Year",
            caption=f"Top 10 flow skills in Technology, Information and Media: {country}",
        )
    )

,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,Telecommunications,Telecommunications,Telecommunications,Broadcasting,Web Development,React.js,React.js,React.js,Web Development
2,Social Media Marketing,Social Media Marketing,Graphic Design,Ghana,JavaScript,JavaScript,Cascading Style Sheets (CSS),JavaScript,Front-End Development
3,Broadcasting,Public Speaking,Web Development,Web Development,Video Editing,Cascading Style Sheets (CSS),Web Development,Python (Programming Language),React.js
4,Public Speaking,Broadcasting,Editing,Journalism,React.js,Python (Programming Language),JavaScript,Web Development,Next.js
5,Research Skills,Web Development,Video Editing,Graphic Design,Media Production,Web Development,Front-End Development,Virtual Assistance,Cascading Style Sheets (CSS)
6,Networking,Graphic Design,Broadcasting,JavaScript,Consumer Services,Graphic Design,Python (Programming Language),Front-End Development,JavaScript
7,Social Media,Team Leadership,JavaScript,Editing,Software Development,Online Advertising,Node.js,Node.js,Python (Programming Language)
8,Team Leadership,Social Media,Blogging,Telecommunications,Ghana,Node.js,HTML,Cascading Style Sheets (CSS),Virtual Assistance
9,Web Development,Networking,Journalism,Node.js,Graphic Design,HTML,Advertising,Artificial Intelligence (AI),Software Development


,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,Telecommunications,Creative Writing,Creative Writing,Web Development,Web Development,Web Content Writing,Cover Letters,Virtual Assistance,Virtual Assistance
2,Editing,Telecommunications,Web Development,Editing,Web Content Writing,Copywriting,Resume Writing,Virtual Administrative Support,Virtual Administrative Support
3,Broadcasting,Web Development,Graphic Design,Writing,Graphic Design,Virtual Assistance,Certified Professional Resume Writer,Search Engine Optimization (SEO),Email Management
4,Web Development,Web Design,Digital Marketing,Digital Marketing,Web Design,Web Design,Resume Review,Email Management,Search Engine Optimization (SEO)
5,Social Media,Graphic Design,Web Content Writing,Graphic Design,Cascading Style Sheets (CSS),Cascading Style Sheets (CSS),Curriculum Vitae (CV),Web Development,Responsive Web Design
6,Creative Writing,Social Media Marketing,Web Design,Web Content Writing,JavaScript,JavaScript,Resumes,Responsive Web Design,Front-End Development
7,Networking,Editing,Telecommunications,Web Design,Creative Writing,Search Engine Optimization (SEO),Screening Resumes,Web Design,Web Design
8,Social Media Marketing,Blogging,Editing,JavaScript,Copywriting,Web Development,Search Engine Optimization (SEO),Web Content Writing,Web Development
9,Blogging,Social Media,Writing,Creative Writing,Search Engine Optimization (SEO),Creative Writing,Applicant Tracking Systems,Front-End Development,Social Media Management


,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,Core Java,Core Java,Core Java,Core Java,Core Java,Core Java,Core Java,Core Java,Amazon Web Services (AWS)
2,SQL,SQL,Python (Programming Language),Python (Programming Language),Java,Java,Java,Amazon Web Services (AWS),Core Java
3,C (Programming Language),Java,C (Programming Language),C (Programming Language),SQL,SQL,REST APIs,Java,Java
4,Java,C (Programming Language),SQL,Data Structures,Python (Programming Language),Amazon Web Services (AWS),Amazon Web Services (AWS),REST APIs,REST APIs
5,Software Development Life Cycle (SDLC),JavaScript,Java,Java,Spring Boot,Spring Boot,SQL,SQL,SQL
6,JavaScript,Python (Programming Language),Data Structures,Amazon Web Services (AWS),C (Programming Language),Python (Programming Language),Spring Boot,React.js,React.js
7,HTML,HTML,JavaScript,SQL,Amazon Web Services (AWS),JavaScript,JavaScript,Spring Boot,Spring Boot
8,Requirements Analysis,Software Development Life Cycle (SDLC),Amazon Web Services (AWS),Spring Boot,JavaScript,REST APIs,Cascading Style Sheets (CSS),JavaScript,Python (Programming Language)
9,C++,Agile Methodologies,Spring Boot,JavaScript,MySQL,Cascading Style Sheets (CSS),Git,Git,Git


,2017,2018,2019,2020,2021,2022,2023,2024,2025
Rank,,,,,,,,,
1,Editing,Editing,Editing,Client Rapport,Agile Application Development,Editing,Search Engine Optimization (SEO),Swahili,Swahili
2,Video Production,Video Production,Python (Programming Language),Exceeding Customer Expectations,Agile Project Management,Virtual Assistance,React.js,Virtual Assistance,Virtual Assistance
3,Broadcasting,Video Editing,Video Production,Personal Development,Kanban,Data Entry,Swahili,Search Engine Optimization (SEO),Data Annotation
4,Video Editing,Social Media Marketing,Telecommunications,Customer Communication,Lean Management,Python (Programming Language),JavaScript,Virtual Administrative Support,Virtual Administrative Support
5,Journalism,Networking,Web Development,Career Management,DevOps,JavaScript,Editing,Python (Programming Language),Search Engine Optimization (SEO)
6,Creative Writing,Creative Writing,Journalism,Design Thinking,Scrum,Web Content Writing,Python (Programming Language),Typing,Web Development
7,Television,Journalism,Video Editing,Active Listening,Agile Methodologies,React.js,Software Development,Data Annotation,React.js
8,Research Skills,Telecommunications,JavaScript,Life Skills,Customer Focused Design,SEO Copywriting,Web Development,React.js,Full-Stack Development
9,Telecommunications,Research Skills,Cloud Computing,Leading Positive Change,Design Thinking,Transcription,Web Content Writing,Data Entry,Software Development


,2017,2018,2019,2020,2021,2022,2023,2024
Rank,,,,,,,,
1,Telecommunications,Telecommunications,Telecommunications,Telecommunications,Afrikaans,Telecommunications,Afrikaans,Afrikaans
2,Wireless Technologies,Wireless Technologies,Afrikaans,Afrikaans,Telecommunications,Computer Literacy,Attention to Detail,Attention to Detail
3,Voice over IP (VoIP),Troubleshooting,Technical Support,Editing,Computer Literacy,Editing,Customer Support,Typing
4,Business Analysis,Business Analysis,Broadcasting,Broadcasting,Consumer Services,Customer Support,Computer Literacy,Service-Level Agreements (SLA)
5,Broadcasting,Social Media Marketing,Troubleshooting,Computer Literacy,Editing,Troubleshooting,Telecommunications,Microsoft Azure
6,Editing,Editing,Computer Literacy,Technical Support,Technical Support,Video Editing,Microsoft Azure,Computer Literacy
7,IT Integration,Software Development Life Cycle (SDLC),SQL,Writing,Video Editing,Amazon Web Services (AWS),Customer Experience,Phone Etiquette
8,Internet Protocol (IP),Broadcasting,Editing,Amazon Web Services (AWS),Electronic Data Capture (EDC),Technical Support,SQL,Customer Engagement
9,SQL,Networking,Wireless Technologies,Customer Experience,Information Technology,SQL,Service-Level Agreements (SLA),Customer Support
